# Kalman filter evaluation

**Goal:** estimate process covariance from training storms only, then evaluate
the constant-velocity Kalman filter on the configured holdout split.

In [ ]:
from pathlib import Path
import pandas as pd
import tomllib

from storm_tracks.data import split_by_storm, validate_disjoint_storm_splits
from storm_tracks.evaluation import evaluate_kalman, summarize_lead_time_errors
from storm_tracks.models import HurricaneKalmanFilter, estimate_noise_covariances

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
with (PROJECT_ROOT / "configs" / "pipeline.toml").open("rb") as file:
    config = tomllib.load(file)
tracks = pd.read_pickle(PROJECT_ROOT / config["paths"]["processed_data"])
splits = split_by_storm(
    tracks,
    train_fraction=config["splits"]["train_fraction"],
    validation_fraction=config["splits"]["validation_fraction"],
    seed=config["splits"]["seed"],
)
validate_disjoint_storm_splits(
    train=splits.train, validation=splits.validation, test=splits.test
)
split_name = config["splits"]["evaluation"]
evaluation_tracks = getattr(splits, split_name)

## Estimate covariance from training trajectories

In [ ]:
process_covariance, observation_covariance = estimate_noise_covariances(
    splits.train,
    min_storm_length=config["model"]["covariance_min_storm_length"],
    sample_size=config["model"]["covariance_sample_size"],
    seed=config["splits"]["seed"],
)
kalman_model = HurricaneKalmanFilter(
    q_base=process_covariance,
    r_base=observation_covariance,
)

## Evaluate the configured split

In [ ]:
kalman_results = evaluate_kalman(
    evaluation_tracks,
    evaluation_tracks["sid"].unique(),
    kalman_model,
    split=split_name,
    lead_times_hours=config["forecast"]["lead_times_hours"],
    min_origin_index=config["forecast"]["min_origin_index"],
    max_origins_per_storm=config["forecast"]["max_origins_per_storm"],
)
summarize_lead_time_errors(kalman_results, split=split_name)

## Next step

Review side-by-side metrics in `05_evaluation_and_visualization.ipynb`.